# 🏭 Air Quality Regression — Capstone Project
## Predicting Continuous Benzene Concentration (`C6H6(GT)`) from the UCI Air Quality Dataset

**Course:** TYBCA — BCA30030 (Unit 3: Regression)  
**Dataset:** [UCI Air Quality Dataset](https://archive.ics.uci.edu/ml/datasets/Air+Quality)  
**Task Type:** Supervised Learning — Regression  
**Target Variable:** `C6H6(GT)` — Ground-truth hourly Benzene concentration (µg/m³)  

---

### Notebook Structure

| Phase | Description |
|-------|-------------|
| **Phase 1** | Setup & Data Ingestion |
| **Phase 2** | Preprocessing & Leakage Prevention |
| **Phase 3** | Exploratory Data Analysis (EDA) |
| **Phase 4** | Supervised ML — Regression Modelling |
| **Phase 5** | Final Evaluation & Model Export |

---
## Phase 1 — Setup & Data Ingestion (Unit 1 & 2)

We begin by importing all required libraries and loading the raw CSV.  
The UCI Air Quality file uses **semicolons (`;`)** as field separators and **commas (`,`)** as decimal marks — a common European CSV convention that Pandas handles via `sep` and `decimal` parameters.

In [ ]:
# Import core data-science libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

warnings.filterwarnings('ignore')
print('\u2705 Core libraries imported successfully.')

In [ ]:
# Import sklearn preprocessing, model-selection, regressors, and metrics
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('\u2705 Scikit-learn modules imported successfully.')

In [ ]:
# Load the UCI Air Quality CSV (European format: sep=';', decimal=',')
df = pd.read_csv('AirQualityUCI.csv', sep=';', decimal=',')
print(f'Raw shape after loading: {df.shape}')
df.head()

In [ ]:
# Drop completely empty trailing columns and trailing empty rows
df.drop(columns=['Unnamed: 15', 'Unnamed: 16'], inplace=True)
df.dropna(how='all', inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Shape after cleanup: {df.shape}  \u2192  Expected \u2248 9,357 rows')

---
## Phase 2 — Preprocessing & Leakage Prevention (Unit 2)

### Why is Preprocessing Order Critical?

In a supervised learning pipeline, **data leakage** occurs when information from the test set inadvertently influences the training process. This leads to over-optimistic metrics that do not generalise to unseen data.

To prevent leakage we follow this strict order:

1. **Replace sentinel values** → 2. **Create datetime index** → 3. **Impute NaNs** → 4. **Define X / y** → 5. **Train/Test Split** → 6. **Outlier handling on train only** → 7. **Scale on train, transform test**

In [ ]:
# Replace the hardware sentinel error code -200 with NaN across the entire DataFrame
df.replace(-200, np.nan, inplace=True)
print(f'Total NaN values after sentinel replacement: {df.isna().sum().sum()}')

In [ ]:
# Combine Date and Time into a single Datetime index and sort chronologically
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H.%M.%S')
df.set_index('Datetime', inplace=True)
df.drop(columns=['Date', 'Time'], inplace=True)
df.sort_index(inplace=True)
print(f'Index range: {df.index.min()} \u2192 {df.index.max()}')
df.head()

### Why Forward-Fill for Time-Series?

Air quality sensors record measurements at hourly intervals. When a sensor reading is missing, the most reasonable assumption is that the concentration hasn't changed drastically since the *previous* valid reading — this is the basis for **Forward-Fill (`.ffill()`)**.  

We follow up with **Backward-Fill (`.bfill()`)** to handle any NaNs at the very start of the series that forward-fill cannot reach.

In [ ]:
# Impute missing values using time-series forward-fill then backward-fill
df.ffill(inplace=True)
df.bfill(inplace=True)
print(f'Remaining NaN values: {df.isna().sum().sum()}')

In [ ]:
# Define target variable y and feature matrix X
target_col = 'C6H6(GT)'
feature_cols = ['PT08.S1(CO)', 'PT08.S2(NMHC)', 'NOx(GT)', 'NO2(GT)', 'T', 'RH']

y = df[target_col].copy()
X = df[feature_cols].copy()

print(f'Feature matrix X shape: {X.shape}')
print(f'Target vector y shape:  {y.shape}')

### Data Leakage Prevention — Split Before Transform

We perform the **Train/Test Split (80/20) BEFORE** any scaling or outlier handling.  
If we were to scale or clip on the full dataset, statistical properties of the test set (its mean, standard deviation, quartiles) would leak into the training pipeline, producing artificially inflated evaluation scores.

In [ ]:
# Train/Test Split (80/20) — BEFORE scaling to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}  |  y_test: {y_test.shape}')

### Why IQR Clipping (Winsorization) Over Dropping Outliers?

Dropping outliers removes data points entirely, which:
- Reduces the training set size, weakening model generalisability.
- Can discard legitimately extreme but valid sensor readings.

**Winsorization** (IQR-based clipping) instead caps extreme values at the whisker bounds (`Q1 \u2212 1.5\u00b7IQR` and `Q3 + 1.5\u00b7IQR`), retaining all observations while limiting the influence of outliers on model fitting.

Critically, the IQR bounds are computed **strictly on `X_train`** and then applied to both `X_train` and `X_test` — this ensures no information leaks from the test partition.

In [ ]:
# Calculate IQR bounds on X_train and apply Winsorization (clipping) to both splits
Q1 = X_train.quantile(0.25)
Q3 = X_train.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

X_train = X_train.clip(lower=lower_bound, upper=upper_bound, axis=1)
X_test  = X_test.clip(lower=lower_bound, upper=upper_bound, axis=1)

print('\u2705 IQR Winsorization applied (bounds computed on training set only).')
print(f'Lower bounds:\n{lower_bound}\n')
print(f'Upper bounds:\n{upper_bound}')

In [ ]:
# Fit StandardScaler on X_train, then transform both X_train and X_test
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('\u2705 StandardScaler fitted on training data and applied to both splits.')
print(f'Train mean \u2248 0: {X_train_scaled.mean(axis=0).round(6)}')
print(f'Train std  \u2248 1: {X_train_scaled.std(axis=0).round(4)}')

---
## Phase 3 — Exploratory Data Analysis (Unit 2)

EDA is performed **after** the train/test split. All statistical summaries and visualisations below are computed on the **training partition** to remain consistent with our leakage-free pipeline.  
We examine:
1. **Univariate Non-graphical** — Central tendency & dispersion
2. **Univariate Graphical** — Histograms, boxplots
3. **Multivariate Non-graphical** — Correlation matrix
4. **Multivariate Graphical** — Heatmap, scatter/regression plot

In [ ]:
# Univariate Non-graphical: Central tendency & dispersion for training features
stats = X_train.agg(['mean', 'median', 'std', 'min', 'max']).T
stats.columns = ['Mean', 'Median', 'Std Dev', 'Min', 'Max']
print('\u2550' * 70)
print('  Measures of Central Tendency & Dispersion (Training Set)')
print('\u2550' * 70)
stats.round(3)

In [ ]:
# Univariate Graphical: Histogram with KDE for the target variable C6H6(GT)
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(y_train, kde=True, bins=50, color='#6C5CE7', edgecolor='white', ax=ax)
ax.set_title('Distribution of Benzene Concentration \u2014 C6H6(GT) (Training Set)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('C6H6(GT) \u2014 \u00b5g/m\u00b3', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.axvline(y_train.mean(), color='#E17055', linestyle='--', lw=2, label=f'Mean = {y_train.mean():.2f}')
ax.axvline(y_train.median(), color='#00B894', linestyle='--', lw=2, label=f'Median = {y_train.median():.2f}')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Univariate Graphical: Boxplots for the scaled training feature matrix
fig, ax = plt.subplots(figsize=(10, 5))
train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_cols)
train_scaled_df.boxplot(
    ax=ax,
    patch_artist=True,
    boxprops=dict(facecolor='#74B9FF', edgecolor='#2D3436'),
    medianprops=dict(color='#D63031', linewidth=2),
    whiskerprops=dict(color='#2D3436'),
    capprops=dict(color='#2D3436'),
    flierprops=dict(marker='o', markerfacecolor='#FDCB6E', markersize=4)
)
ax.set_title('Feature Distributions After Scaling (Training Set)', fontsize=14, fontweight='bold')
ax.set_ylabel('Scaled Value', fontsize=12)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Multivariate Non-graphical: Pearson correlation matrix (training features + target)
train_full = X_train.copy()
train_full[target_col] = y_train
corr_matrix = train_full.corr(method='pearson').round(3)
print('\u2550' * 70)
print('  Pearson Correlation Matrix (Training Set)')
print('\u2550' * 70)
corr_matrix

In [ ]:
# Multivariate Graphical: Seaborn Correlation Heatmap
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    square=True,
    cbar_kws={'shrink': 0.8},
    ax=ax
)
ax.set_title('Pearson Correlation Heatmap (Training Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Multivariate Graphical: Regplot — PT08.S2(NMHC) vs C6H6(GT)
fig, ax = plt.subplots(figsize=(9, 6))
sns.regplot(
    x=X_train['PT08.S2(NMHC)'],
    y=y_train,
    scatter_kws={'alpha': 0.3, 's': 15, 'color': '#0984E3'},
    line_kws={'color': '#D63031', 'linewidth': 2},
    ax=ax
)
ax.set_title('PT08.S2(NMHC) vs C6H6(GT) \u2014 Scatter + Regression Line',
             fontsize=14, fontweight='bold')
ax.set_xlabel('PT08.S2(NMHC) \u2014 Sensor Response', fontsize=12)
ax.set_ylabel('C6H6(GT) \u2014 Benzene (\u00b5g/m\u00b3)', fontsize=12)
plt.tight_layout()
plt.show()

---
## Phase 4 — Supervised Machine Learning: Regression (Unit 3)

We train and evaluate **five** regression models strictly from the BCA30030 Unit 3 syllabus:

| # | Model | Key Hyperparameters |
|---|-------|---------------------|
| 1 | Multiple Linear Regression | \u2014 (OLS baseline) |
| 2 | Ridge Regression | \u03b1 = 1.0 (L2 penalty) |
| 3 | Lasso Regression | \u03b1 = 0.1 (L1 penalty) |
| 4 | Support Vector Regressor | kernel='rbf', C=100, \u03b5=0.1 |
| 5 | Random Forest Regressor | n_estimators=200, max_depth=15 |

### Evaluation Metrics (Unit 3)

| Metric | Formula | Purpose |
|--------|---------|---------|
| **MAE** | $\\frac{1}{n}\\sum|y_i - \\hat{y}_i|$ | Average absolute error |
| **MSE** | $\\frac{1}{n}\\sum(y_i - \\hat{y}_i)^2$ | Penalises large errors |
| **RMSE** | $\\sqrt{MSE}$ | Error in target units |
| **R\u00b2** | $1 - \\frac{SS_{res}}{SS_{tot}}$ | Variance explained |
| **Adj. R\u00b2** | $1 - \\frac{(1-R^2)(n-1)}{n-k-1}$ | R\u00b2 adjusted for # features |

In [ ]:
# Helper function to compute all Unit 3 regression metrics
def evaluate_model(y_true, y_pred, n_features):
    """Return a dict of MAE, MSE, RMSE, R\u00b2, Adjusted R\u00b2."""
    n = len(y_true)
    k = n_features
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    adj_r2 = 1 - ((1 - r2) * (n - 1)) / (n - k - 1)
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R\u00b2': r2, 'Adj. R\u00b2': adj_r2}

# Dictionary to store results for all models
results = {}
predictions = {}

n_features = X_train_scaled.shape[1]
print(f'Number of features (k): {n_features}')

### Model 1 — Multiple Linear Regression

The OLS (Ordinary Least Squares) baseline fits a hyperplane that minimises the sum of squared residuals. It assumes a **linear relationship** between features and target, **no multicollinearity**, and **homoscedastic** residuals.

In [ ]:
# Model 1A: Fit Multiple Linear Regression and predict on test data
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
predictions['Linear Regression'] = y_pred_lr
print('\u2705 Multiple Linear Regression fitted and predictions generated.')

In [ ]:
# Model 1B: Evaluate Multiple Linear Regression metrics
results['Linear Regression'] = evaluate_model(y_test, y_pred_lr, n_features)
print('\u2550' * 55)
print('  Multiple Linear Regression \u2014 Evaluation Metrics')
print('\u2550' * 55)
for metric, value in results['Linear Regression'].items():
    print(f'  {metric:>10s} : {value:.4f}')

In [ ]:
# Model 1C: Residual scatter plot — True vs Predicted with 45-degree reference line
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred_lr, alpha=0.35, s=15, color='#0984E3', label='Predictions')
lims = [min(y_test.min(), y_pred_lr.min()) - 1, max(y_test.max(), y_pred_lr.max()) + 1]
ax.plot(lims, lims, '--', color='#D63031', lw=2, label='Perfect Prediction (45\u00b0)')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Actual C6H6(GT)', fontsize=12)
ax.set_ylabel('Predicted C6H6(GT)', fontsize=12)
ax.set_title('Multiple Linear Regression \u2014 Actual vs Predicted', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Model 2 — Ridge Regression (L2 Regularisation)

Ridge adds an **L2 penalty** ($\\alpha \\sum \\beta_j^2$) to the OLS cost function.  
This shrinks coefficients toward zero without eliminating any, which is effective when features are **correlated** (multicollinearity) — as is common with co-located air-quality sensors.

In [ ]:
# Model 2A: Fit Ridge Regression and predict on test data
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)
predictions['Ridge Regression'] = y_pred_ridge
print('\u2705 Ridge Regression fitted and predictions generated.')

In [ ]:
# Model 2B: Evaluate Ridge Regression metrics
results['Ridge Regression'] = evaluate_model(y_test, y_pred_ridge, n_features)
print('\u2550' * 55)
print('  Ridge Regression \u2014 Evaluation Metrics')
print('\u2550' * 55)
for metric, value in results['Ridge Regression'].items():
    print(f'  {metric:>10s} : {value:.4f}')

In [ ]:
# Model 2C: Residual scatter plot — True vs Predicted with 45-degree reference line
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred_ridge, alpha=0.35, s=15, color='#00B894', label='Predictions')
lims = [min(y_test.min(), y_pred_ridge.min()) - 1, max(y_test.max(), y_pred_ridge.max()) + 1]
ax.plot(lims, lims, '--', color='#D63031', lw=2, label='Perfect Prediction (45\u00b0)')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Actual C6H6(GT)', fontsize=12)
ax.set_ylabel('Predicted C6H6(GT)', fontsize=12)
ax.set_title('Ridge Regression \u2014 Actual vs Predicted', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Model 3 — Lasso Regression (L1 Regularisation)

Lasso adds an **L1 penalty** ($\\alpha \\sum |\\beta_j|$).  
Unlike Ridge, L1 can drive coefficients to **exactly zero**, performing automatic **feature selection**. We use a smaller `alpha=0.1` because higher values can over-shrink and degrade performance on this dataset.

In [ ]:
# Model 3A: Fit Lasso Regression and predict on test data
lasso = Lasso(alpha=0.1, random_state=42, max_iter=10000)
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_test_scaled)
predictions['Lasso Regression'] = y_pred_lasso
print('\u2705 Lasso Regression fitted and predictions generated.')

In [ ]:
# Model 3B: Evaluate Lasso Regression metrics
results['Lasso Regression'] = evaluate_model(y_test, y_pred_lasso, n_features)
print('\u2550' * 55)
print('  Lasso Regression \u2014 Evaluation Metrics')
print('\u2550' * 55)
for metric, value in results['Lasso Regression'].items():
    print(f'  {metric:>10s} : {value:.4f}')

In [ ]:
# Model 3C: Residual scatter plot — True vs Predicted with 45-degree reference line
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred_lasso, alpha=0.35, s=15, color='#E17055', label='Predictions')
lims = [min(y_test.min(), y_pred_lasso.min()) - 1, max(y_test.max(), y_pred_lasso.max()) + 1]
ax.plot(lims, lims, '--', color='#D63031', lw=2, label='Perfect Prediction (45\u00b0)')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Actual C6H6(GT)', fontsize=12)
ax.set_ylabel('Predicted C6H6(GT)', fontsize=12)
ax.set_title('Lasso Regression \u2014 Actual vs Predicted', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Model 4 — Support Vector Regressor (SVR)

SVR finds a function within an **\u03b5-insensitive tube** — predictions within \u03b5 of the true value incur zero loss. Points outside the tube become **support vectors**.  
The **RBF kernel** projects data into a higher-dimensional space, allowing SVR to model non-linear relationships. `C=100` gives a wider regularisation margin, and `epsilon=0.1` defines the tube width.

In [ ]:
# Model 4A: Fit SVR (RBF kernel) and predict on test data
svr = SVR(kernel='rbf', C=100, epsilon=0.1)
svr.fit(X_train_scaled, y_train)
y_pred_svr = svr.predict(X_test_scaled)
predictions['SVR'] = y_pred_svr
print('\u2705 Support Vector Regressor fitted and predictions generated.')

In [ ]:
# Model 4B: Evaluate SVR metrics
results['SVR'] = evaluate_model(y_test, y_pred_svr, n_features)
print('\u2550' * 55)
print('  Support Vector Regressor \u2014 Evaluation Metrics')
print('\u2550' * 55)
for metric, value in results['SVR'].items():
    print(f'  {metric:>10s} : {value:.4f}')

In [ ]:
# Model 4C: Residual scatter plot — True vs Predicted with 45-degree reference line
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred_svr, alpha=0.35, s=15, color='#6C5CE7', label='Predictions')
lims = [min(y_test.min(), y_pred_svr.min()) - 1, max(y_test.max(), y_pred_svr.max()) + 1]
ax.plot(lims, lims, '--', color='#D63031', lw=2, label='Perfect Prediction (45\u00b0)')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Actual C6H6(GT)', fontsize=12)
ax.set_ylabel('Predicted C6H6(GT)', fontsize=12)
ax.set_title('SVR (RBF) \u2014 Actual vs Predicted', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Model 5 — Random Forest Regressor

Random Forest is an **ensemble** of Decision Trees trained on bootstrapped samples with random feature subsets (bagging).  
It reduces variance via averaging, handles non-linearity naturally, and is robust to outliers. We cap `max_depth=15` to prevent overfitting on ~7,500 training samples.

In [ ]:
# Model 5A: Fit Random Forest Regressor and predict on test data
rf = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)
predictions['Random Forest'] = y_pred_rf
print('\u2705 Random Forest Regressor fitted and predictions generated.')

In [ ]:
# Model 5B: Evaluate Random Forest metrics
results['Random Forest'] = evaluate_model(y_test, y_pred_rf, n_features)
print('\u2550' * 55)
print('  Random Forest Regressor \u2014 Evaluation Metrics')
print('\u2550' * 55)
for metric, value in results['Random Forest'].items():
    print(f'  {metric:>10s} : {value:.4f}')

In [ ]:
# Model 5C: Residual scatter plot — True vs Predicted with 45-degree reference line
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred_rf, alpha=0.35, s=15, color='#FDCB6E', edgecolors='#2D3436', linewidths=0.3, label='Predictions')
lims = [min(y_test.min(), y_pred_rf.min()) - 1, max(y_test.max(), y_pred_rf.max()) + 1]
ax.plot(lims, lims, '--', color='#D63031', lw=2, label='Perfect Prediction (45\u00b0)')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Actual C6H6(GT)', fontsize=12)
ax.set_ylabel('Predicted C6H6(GT)', fontsize=12)
ax.set_title('Random Forest \u2014 Actual vs Predicted', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## Phase 5 — Final Evaluation & Model Export

We consolidate the performance of all five models into a comparative table and a grouped bar chart, then export the champion model and the scaler for future deployment or inference.

In [ ]:
# Compile RMSE and R-Squared of all 5 models into a DataFrame, sorted by best R-Squared
comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'RMSE':  [results[m]['RMSE'] for m in results],
    'R\u00b2':    [results[m]['R\u00b2']   for m in results],
    'Adj. R\u00b2': [results[m]['Adj. R\u00b2'] for m in results]
})
comparison.sort_values(by='R\u00b2', ascending=False, inplace=True)
comparison.reset_index(drop=True, inplace=True)
print('\u2550' * 60)
print('  Model Comparison \u2014 Sorted by R\u00b2 (Descending)')
print('\u2550' * 60)
comparison.round(4)

In [ ]:
# Grouped bar chart comparing RMSE and R-Squared across all models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#0984E3', '#00B894', '#E17055', '#6C5CE7', '#FDCB6E']

# RMSE bar chart
bars1 = axes[0].barh(comparison['Model'], comparison['RMSE'], color=colors, edgecolor='#2D3436')
axes[0].set_xlabel('RMSE', fontsize=12)
axes[0].set_title('RMSE Comparison (Lower is Better)', fontsize=13, fontweight='bold')
axes[0].invert_yaxis()
for bar, val in zip(bars1, comparison['RMSE']):
    axes[0].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=10, fontweight='bold')

# R-Squared bar chart
bars2 = axes[1].barh(comparison['Model'], comparison['R\u00b2'], color=colors, edgecolor='#2D3436')
axes[1].set_xlabel('R\u00b2', fontsize=12)
axes[1].set_title('R\u00b2 Comparison (Higher is Better)', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()
for bar, val in zip(bars2, comparison['R\u00b2']):
    axes[1].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Export the Champion Model and StandardScaler using joblib
champion_name = comparison.iloc[0]['Model']
print(f'\U0001F3C6 Champion Model: {champion_name}')

# Map model name to fitted object
model_map = {
    'Linear Regression': lr,
    'Ridge Regression': ridge,
    'Lasso Regression': lasso,
    'SVR': svr,
    'Random Forest': rf
}

champion_model = model_map[champion_name]

joblib.dump(champion_model, 'air_quality_regressor.pkl')
joblib.dump(scaler, 'scaler.pkl')

print(f'\u2705 Saved: air_quality_regressor.pkl  ({champion_name})')
print(f'\u2705 Saved: scaler.pkl  (StandardScaler)')

---
## \ud83d\udcdd Summary & Conclusions

| Step | What was done |
|------|---------------|
| **Data Ingestion** | Loaded UCI Air Quality CSV (European format), dropped empty columns/rows |
| **Preprocessing** | Replaced \u2212200 sentinels \u2192 NaN, created Datetime index, forward/backward filled |
| **Leakage Prevention** | Train/Test split **before** outlier handling & scaling |
| **Outlier Handling** | IQR Winsorization with bounds derived from training set only |
| **Scaling** | StandardScaler fit on train, transformed both splits |
| **EDA** | Univariate & multivariate analysis on training partition |
| **Modelling** | 5 regressors: Linear, Ridge, Lasso, SVR, Random Forest |
| **Evaluation** | MAE, MSE, RMSE, R\u00b2, Adjusted R\u00b2 \u2014 comparative table & chart |
| **Export** | Champion model + scaler serialised via `joblib` |

---
*Notebook prepared for TYBCA BCA30030 Capstone \u2014 Regression on UCI Air Quality Dataset.*